In [0]:
%pip install -q databricks-langchain "langgraph>=0.3,<0.4" databricks-agents "psycopg[binary]>=3.1.0" pydantic databricks-ai-search

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# Fraud Triage Copilot Agent
**Step 5:** Fraud Triage Copilot (Layer 4) + Lakebase FGAC

- FM endpoint: `databricks-claude-sonnet-5-5`
- Tools (since 2026-10-21 FGAC hardening): all agent-side and Postgres-governed via `LakebaseCases(pg_role='fraud_triage_agent')`: `get_transaction_risk` (calibrated confidence), `explain_top_signals`, `get_customer_history`, `create_fraud_case`. The UC function versions are deprecated and not registered.
- Registered: `workshop_guy_catalog.fe_bar_a0.fraud_triage_agent` v1 @champion
- Experiment: `/Users/alex.owen@databricks.com/fe_bar_fraud_triage_agent`

In [0]:
import sys, json, time
sys.path.insert(0, "/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo")

import mlflow
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/alex.owen@databricks.com/fe_bar_fraud_triage_agent")

from fraud_triage_agent import FraudTriageAgent
from mlflow.types.responses import ResponsesAgentRequest

agent = FraudTriageAgent()
print("Agent ready.")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
/local_disk0/.ephemeral_nfs/envs/pythonEnv-adb9d85a-e300-4444-bab3-dfbdb57bc928/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026/10/23 23:25:40 WARNING mlflow.utils.autologging_utils: MLflow langchain autologging is known to be compatible with 1.0.0 <= langchain, but the installed version is 0.3.30. If you encounter errors during autologging, try upgrading / downgrading langchain to a compatible version, or 

Agent ready.


In [0]:
# E.1: Why was TXN-0234209 flagged?
request = ResponsesAgentRequest(
    input=[{"role": "user", "content": "Why was TXN-0234209 flagged and what should we do?"}]
)
t0 = time.time()
response = agent.predict(request)
print(f"Latency: {time.time()-t0:.1f}s")
print(response.output[0].content[0]["text"])

# FGAC hardening Item 5 VERIFY: trace span table for this run.
trace_id = mlflow.get_last_active_trace_id()
trace = mlflow.get_trace(trace_id)
print(f"\n=== Trace span table  trace_id={trace_id} ===")
rows = []
for s in trace.data.spans:
    if not str(s.span_type).upper().endswith("TOOL"):
        continue
    out = s.outputs
    if hasattr(out, "content"): out = out.content
    if isinstance(out, dict) and "content" in out: out = out["content"]
    try:
        payload = json.loads(out) if isinstance(out, str) else {}
    except Exception:
        payload = {}
    rows.append((s.name, payload.get("_implementation", "(no provenance)"), ", ".join(payload.get("_objects_read", [])), (s.end_time_ns - s.start_time_ns) / 1e6))
print(f"{'tool span':<24} {'implementation':<42} {'ms':>8}  objects read")
print("-" * 150)
for name, impl, objs, ms in rows:
    print(f"{name:<24} {impl:<42} {ms:>8.1f}  {objs}")
assert rows, "no tool spans found"
assert all(r[1] == "agent_python:postgres:fraud_triage_agent" for r in rows), "a tool ran outside Postgres governance"
print(f"\n{len(rows)} tool spans, all Postgres-governed (fraud_triage_agent); no UC function tool was called.")

2026/10/23 23:25:44 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-adb9d85a-e300-4444-bab3-dfbdb57bc928/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...escalate it, tell me."}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Latency: 17.2s
**TXN-0234209 was flagged as high risk (score 0.984). The model recommends blocking it, and an open case already exists.**

**Why it was flagged**
- **New device (SHAP +2.57, the strongest signal):** The customer normally uses about 5 devices in 30 days, but this one is new.
- **Distance from home (SHAP +1.87):** The transaction was 450.6 km from the customer's home region, US-East.
- **Online transaction (SHAP +1.20):** Online transactions add risk in this model.
- **Amount:** $323.96 is about 3x the customer's 30-day average of $100.44. This isn't one of the top signals, but it adds context.

**Recommended action:** Block.

**Confidence:** Medium (observed precision 0.50 in block_band, n=288). In past cases in this band, only about half of the blocks turned out to be fraud. The score is high, but the confidence is not.

**Routed queue:** auto_block

**Customer context (CUST-08669)**
- **Profile:** The customer has 40 months of tenure (the tool didn't give the unit) and

Trace(trace_id='tr-d04a4f505c11e1fca071487a6fb8abd6')

In [0]:
# E.2: Open a case
request_act = ResponsesAgentRequest(
    input=[{"role": "user", "content": "Open a case for TXN-0234209 and route it."}]
)
t0 = time.time()
response_act = agent.predict(request_act)
print(f"Latency: {time.time()-t0:.1f}s")
print(response_act.output[0].content[0]["text"])
# Case 2319d4b7-bea3-412b-ba6b-8325e1a7a71d (opened in Step 5) is still open, so the
# idempotent create_fraud_case should return that same case_id.
from lakebase_cases import LakebaseCases
_lc = LakebaseCases()
for c in _lc.get_cases_for_txn("TXN-0234209"):
    print(f"case {c['case_id']} status={c['status']} queue={c['assigned_queue']} created_by={c['created_by']} db_inserted_by={c['db_inserted_by']}")
_lc.close()

/local_disk0/.ephemeral_nfs/envs/pythonEnv-adb9d85a-e300-4444-bab3-dfbdb57bc928/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...TbyUWQXfBQ0cFB7GAE='}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-adb9d85a-e300-4444-bab3-dfbdb57bc928/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...the existing case ID."}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Latency: 12.2s
I haven't created a new case for TXN-0234209. An open case already exists for it, and I didn't want to add a duplicate.

**Existing case:** `2319d4b7-bea3-412b-ba6b-8325e1a7a71d`. Its status is open and its queue is `auto_block`, which matches the model's routing.

## TXN-0234209: $323.96, CUST-08669

**Why flagged**
- **New device (SHAP +2.57):** The customer normally uses 5 devices in 30 days, and this one is new.
- **Distance from home (SHAP +1.87):** The transaction was 450.6 km from the home region, US-East.
- **Online transaction (SHAP +1.20):** Online transactions score as higher risk.

**Risk score:** 0.984

**Recommended action:** Block.

**Confidence:** Medium (observed precision 0.50 in block_band, n=288). Roughly half of transactions in this band turned out to be fraud, so this is not a high-certainty block.

**Routed queue:** `auto_block`

**Customer context**
- The customer is in the low risk segment, with 40 months of tenure.
- Over the last 30 days they m

Trace(trace_id='tr-c4dfac39a3aa18c64a6fe76a7d63c18b')

In [0]:
# E.3: Ask again - should return same case_id
response_idem = agent.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": "Open a case for TXN-0234209 and route it."}]
))
print(response_idem.output[0].content[0]["text"][:1000])
from lakebase_cases import LakebaseCases
lc = LakebaseCases()
open_cases = [c for c in lc.get_cases_for_txn('TXN-0234209') if c['status'] != 'closed']
print(f"\nOpen cases after 2 requests: {len(open_cases)} (expected: 1)")
for c in open_cases:
    print(f"  case_id={c['case_id']}")
lc.close()

/local_disk0/.ephemeral_nfs/envs/pythonEnv-adb9d85a-e300-4444-bab3-dfbdb57bc928/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...tfzHLa8pxpmv+u8AhgB'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-adb9d85a-e300-4444-bab3-dfbdb57bc928/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...model's block action."}], input_type=list])
  return self.__pydantic_serializer__.to_python(


I haven't created a new case for TXN-0234209. An open case already exists for it, and I didn't want to create a duplicate.

**Existing case:** `2319d4b7-bea3-412b-ba6b-8325e1a7a71d`, status open, queue auto_block.

**Why it was flagged**
- **New device (SHAP +2.57):** The customer normally uses 5 devices in 30 days, and this one is new.
- **Distance from home (SHAP +1.87):** The transaction was 450.6 km from the home region, US-East.
- **Online transaction (SHAP +1.20):** Online transactions score as higher risk.

**Transaction:** $323.96 on CUST-08669, risk score 0.984.

**Recommended action:** Block.

**Confidence:** Medium (observed precision 0.50 in block_band, n=288). Only about half of past transactions in this band were confirmed fraud, so the high score overstates how reliable a block is.

**Routed queue:** auto_block.

**Customer context (CUST-08669)**
- The customer is low risk, with 40 months of tenure.
- The customer has 3 prior cases, and all 3 were confirmed fraud. That s

Trace(trace_id='tr-897a760ca641f6f6986521c41e91b1ae')

## E.5 Batch Triage - Top 5 Block Transactions
| txn_id | risk_score | case_id |
|--------|-----------|--------|
| TXN-0147293 | 0.993762 | 82b12d25-bf88-47cb-b0a4-f5a5b62f349d |
| TXN-0288343 | 0.992881 | 863334d8-909e-4348-8be4-772d0c7f6e72 |
| TXN-0221786 | 0.992875 | c37587c0-5091-4fd7-a656-c9df9417d0e3 |
| TXN-0417769 | 0.992698 | 98edced4-9dc7-47a3-bbf3-1694bb7b1b43 |
| TXN-0432087 | 0.992657 | 98adcae4-cec6-46a1-a2c5-1ecd84c299ad |

In [0]:
# E.6: Guardrails
# Unknown txn
resp_unknown = agent.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": "Why was TXN-9999999 flagged?"}]
))
print("--- Unknown txn ---")
print(resp_unknown.output[0].content[0]["text"][:500])

# Approve band should not be blocked
resp_approve = agent.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": "Block TXN-0278800 immediately."}]
))
print("\n--- Approve-band ---")
print(resp_approve.output[0].content[0]["text"][:500])

--- Unknown txn ---
I couldn't find **TXN-9999999**. The risk lookup and the signal explanation both came back empty. That means I have no score, recommended action, confidence level, routed queue, or SHAP signals for it, and I can't say why it was flagged. I also can't pull customer context, because I don't know which customer it belongs to.

Here is what you can try:
- **Check the ID.** Look for typos, a missing or extra digit, or a different prefix format.
- **Check whether it has been scored yet.** A very recen


/local_disk0/.ephemeral_nfs/envs/pythonEnv-adb9d85a-e300-4444-bab3-dfbdb57bc928/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...re taking any action."}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-adb9d85a-e300-4444-bab3-dfbdb57bc928/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...h option do you want?"}], input_type=list])
  return self.__pydantic_serializer__.to_python(



--- Approve-band ---
I haven't blocked TXN-0278800, because the model scored it in the approve band and I shouldn't push an approve-band transaction into block. I haven't created a case yet.

**Why it was flagged**
- **New device (SHAP +1.76):** This is the strongest signal. The customer normally uses 5 devices in 30 days.
- **Online transaction (SHAP +0.99):** Online carries higher risk.
- **Distance from home (SHAP +0.62):** The transaction was 575.6 km from the customer's EU home region.

**Model assessment**
- *


[Trace(trace_id='tr-757712b2985159a008e158f24eb5d78c'), Trace(trace_id='tr-427e87bbd00c32c4bf3a496c6442da75')]

In [0]:
# D.3: Log and register to UC
# GUARD (2026-10-21 hardening): v1 @champion was registered in Step 5 with the now-deprecated UC function
# tools. Re-registering creates v2 and moves @champion, so it runs only when explicitly enabled.
# Safety flag: False = no new agent version is logged or registered, @champion is not moved.
REGISTER_AGENT = False
from mlflow.tracking import MlflowClient
FULL_NAME = "workshop_guy_catalog.fe_bar_a0.fraud_triage_agent"
mc = MlflowClient(registry_uri="databricks-uc")

if not REGISTER_AGENT:
    v = mc.get_model_version_by_alias(FULL_NAME, "champion")
    print(f"Registration skipped (REGISTER_AGENT=False). Current @champion: v{v.version}, run {v.run_id}. It still embeds the pre-hardening UC-function tools.")
else:
    from mlflow.models.resources import DatabricksServingEndpoint
    resources = [
        DatabricksServingEndpoint(endpoint_name="databricks-claude-sonnet-5-5"),
    ]  # read tools are agent-side Postgres tools now; no UC function resources
    with mlflow.start_run(run_name="fraud_triage_agent_v1"):
        info = mlflow.pyfunc.log_model(
            name="agent",
            python_model="/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/fraud_triage_agent.py",
            resources=resources,
            input_example={"input": [{"role": "user", "content": "Why was TXN-0234209 flagged?"}]},
            pip_requirements=["mlflow>=3.1", "databricks-langchain", "langgraph>=0.3,<0.4",
                              "databricks-agents", "pydantic>=2", "psycopg[binary]>=3.1.0", "databricks-ai-search"],
            registered_model_name=FULL_NAME,
        )
    mc.set_registered_model_alias(FULL_NAME, "champion", info.registered_model_version)
    print(f"Registered: {FULL_NAME} v{info.registered_model_version} @champion")

Registration skipped (REGISTER_AGENT=False). Current @champion: v2, run 97bc6c9b4e654abab627cd25ee93fa53. It still embeds the pre-hardening UC-function tools.
